# 1022 35일차

## 1. 결과 통합 노드

fan-in 노드에서 앞 노드들의 결과를 LLM에 넘겨 하나의 답으로 정리하는 방법 (`lg09_fanout_fanin3`)

결과를 문자열로 이어 붙이기만 하면 겹치는 내용이 그대로 남고 출처마다 따로 읽어야 해서, LLM이 두 결과를 합쳐 답을 새로 쓰게 함

1. 문자열 이어 붙이기 : merge 노드가 결과를 그대로 합침. LLM을 부르지 않음
2. LLM 통합 : 두 결과를 프롬프트에 넣고 LLM이 최종 답을 씀

- 프롬프트에서 결과마다 `[웹 검색 결과]`, `[AI 분석 결과]`처럼 이름을 붙여 구분
- 어느 결과를 우선할지, 어떻게 나눠 설명할지를 프롬프트에 적으면 답의 구성이 그 지시를 따름
- `llm.invoke`가 돌려준 값은 이미 `AIMessage`라 변환 없이 `messages`에 돌려줌
- 통합 노드는 앞 결과를 모두 프롬프트에 넣어서 입력 토큰이 크게 늘어남

## 2. 트리 계열 모델

컬럼 값을 기준으로 데이터를 계속 둘로 나눠서(분기) 예측하는 모델 (`m01_feature_importances01`, `m01_feature_importances02`)

표 형태 데이터를 딥러닝보다 적은 데이터와 짧은 시간으로 학습할 수 있고, 어느 컬럼으로 나눴는지 남아서 컬럼 중요도를 볼 수 있음

1. `DecisionTree` : 트리 하나. `컬럼 <= 값` 같은 질문으로 데이터를 나누고, 더 나누지 않는 끝(리프)에서 분류는 가장 많은 클래스, 회귀는 평균값으로 예측
2. `RandomForest` : 데이터와 컬럼을 무작위로 뽑아 트리 여러 개를 따로 만들고, 분류는 투표·회귀는 평균으로 합침 (배깅)
3. `GradientBoosting` : 트리를 순서대로 만들고, 다음 트리가 앞 트리들이 남긴 오차를 줄이도록 학습 (부스팅)
4. `XGBoost` : 그래디언트 부스팅을 빠르게 구현한 라이브러리. 과적합을 막는 규제가 들어 있음

- 분기 기준은 나눈 뒤 손실(분류는 불순도, 회귀는 오차)이 가장 많이 줄어드는 컬럼과 값
- 트리 하나는 깊어질수록 train에 과적합되기 쉬움. 트리 여러 개를 합치면(앙상블) 과적합이 줄어듦
- 값의 크기 비교로만 나눠서 스케일링을 해도 결과가 같음
- 분류는 `…Classifier`, 회귀는 `…Regressor`
- sklearn 모델은 `compile` 없이 `fit`으로 훈련

## 3. feature_importances_

트리 계열 모델이 학습한 뒤 컬럼마다 매긴 중요도 (`m01_feature_importances01`, `m01_feature_importances02`)

어떤 컬럼이 예측에 많이 쓰였는지 보고, 컬럼을 빼거나 새로 만들어 성능을 올리는 데(feature engineering) 씀

1. `feature_importances_` : `fit`한 뒤에 생기는 속성. 값은 컬럼 순서대로 나오고 합은 1
2. `model.score(x_test, y_test)` : 분류 모델은 acc, 회귀 모델은 r2

- 모델에 넣는 파라미터가 아님. 이름 끝의 `_`가 학습 후 생기는 값이라는 표시
- 같은 데이터라도 모델마다 중요도 값이 다름

#### cf) r2 음수

- r2가 0이면 평균값 하나로 예측한 것과 같고, 음수면 그보다 못함
- 트리 하나(`DecisionTree`)는 train에 과적합되기 쉬워서 test r2가 음수로 나올 수 있음

#### cf) 회귀의 stratify

- `stratify=y`는 클래스 비율을 맞춰 나누는 옵션이라 y가 연속값인 회귀 데이터에는 쓰지 않음

## 4. XGBoost 중요도 기준

XGBoost가 컬럼 중요도를 매기는 기준과 그 값을 꺼내는 방법 (`m01_feature_importances03_gain`, `m01_feature_importances04_graph`)

중요도를 무엇으로 셀지에 따라 컬럼 순위가 달라질 수 있어서, 기준을 정해서 봄

1. weight : 그 컬럼이 분기에 쓰인 횟수. 그래프에서는 F score라고 부름
2. gain : 그 컬럼으로 분기할 때 손실이 줄어든 양의 평균
3. `get_booster().get_score(importance_type=…)` : 기준별 값을 `{컬럼: 값}` dict로 꺼냄
4. `plot_importance(model, importance_type=…)` : 기준별 값을 큰 순서대로 가로 막대그래프로 그림

- XGB의 `feature_importances_`는 gain을 합이 1이 되게 나눈 값
- `plot_importance`의 `importance_type` 기본값은 `'weight'`
- 컬럼 이름 없이 numpy로 학습하면 키가 `f0`, `f1` …
- F score는 분기 횟수라서 분류 평가지표 F1 score와는 다른 값